**Create a Databricks pipeline to process bank transaction data using:**

Bronze → Silver → Gold (Medallion Architecture)

PySpark + Delta Lake

**Bankreceives daily transaction data from core banking system.**
We need to:

Load raw data into Bronze layer

Clean & transform data into Silver layer

Create business aggregates in Gold layer

Make data ready for reporting

In [0]:
# Retrieve storage account key from Azure Key Vault
storage_account_key = dbutils.secrets.get(scope="adb-secret-scope-dev", key="gen2key")

# Configure Azure storage account access
spark.conf.set(
    "fs.azure.account.key.dataengeus2dev.dfs.core.windows.net",
    storage_account_key
)

# Read CSV file with header
diamonds_df = (spark.read
  .format("csv")
  .option("header", "true")        # ✅ important
  .option("inferSchema", "true")   # ✅ optional but recommended
  .option("mode", "PERMISSIVE")
  .load("abfss://bronze@dataengeus2dev.dfs.core.windows.net/transactions/")
)

display(diamonds_df)


In [0]:
# scopes = dbutils.secrets.listScopes()
# for scope in scopes:
#     print("Secret scope name is : ", scope.name)
                                                                 

dbutils.secrets.listScopes()

In [0]:
%sql
CREATE TABLE databricks_dataeng_dev_eus2.dev.silver_transactions
USING DELTA;

In [0]:
%sql
CREATE TABLE databricks_dataeng_dev_eus2.dev.bronze_transactions
USING DELTA;


**Step 3: Create Silver Pipeline (Data Cleaning)**
🎯 Goal:

Remove null amount

Remove duplicates

Convert data types

In [0]:
from pyspark.sql.functions import col

bronze_df = spark.read.format("parquet").load("abfss://bronze@dataengeus2dev.dfs.core.windows.net/transactions/")

silver_df = bronze_df.filter(col("amount").isNotNull()) \
                     .dropDuplicates()

silver_df.write.format("delta").mode("overwrite").save("abfss://bronze@dataengeus2dev.dfs.core.windows.net/transactions/")
